## Saliency Map

### Basic Saliency Map (Vanilla Gradients – PyTorch)

In [2]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from PIL import Image

# Model
model = models.resnet18(pretrained=True)
model.eval()

# Image
img = Image.open("image.jpg").convert("RGB")
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

x = transform(img).unsqueeze(0)
x.requires_grad_()

# Forward
output = model(x)
class_idx = output.argmax()

# Backward
model.zero_grad()
output[0, class_idx].backward()

# Saliency
saliency = x.grad.abs().max(dim=1)[0]

plt.imshow(saliency[0].cpu(), cmap="hot")
plt.axis("off")
plt.title("Saliency Map (Vanilla Gradient)")
plt.show()


ModuleNotFoundError: No module named 'torch'

### Smoothed Saliency Map (Noise Averaging)

In [1]:
import numpy as np

def smooth_saliency(model, x, class_idx, n_samples=20, noise_level=0.1):
    saliency = 0
    for _ in range(n_samples):
        noise = noise_level * torch.randn_like(x)
        noisy_x = (x + noise).requires_grad_()
        output = model(noisy_x)
        model.zero_grad()
        output[0, class_idx].backward()
        saliency += noisy_x.grad.abs()
    return saliency / n_samples

saliency = smooth_saliency(model, x, class_idx)
saliency = saliency.max(dim=1)[0]

plt.imshow(saliency[0].cpu(), cmap="hot")
plt.axis("off")
plt.title("Smoothed Saliency Map")
plt.show()


NameError: name 'model' is not defined

### Guided Backpropagation (PyTorch)

In [ ]:
from torch.nn import ReLU

def guided_relu_hook(module, grad_in, grad_out):
    return (torch.clamp(grad_in[0], min=0),)

for module in model.modules():
    if isinstance(module, ReLU):
        module.register_backward_hook(guided_relu_hook)

x.requires_grad_()
output = model(x)
output[0, class_idx].backward()

saliency = x.grad.abs().max(dim=1)[0]

plt.imshow(saliency[0].cpu(), cmap="hot")
plt.axis("off")
plt.title("Guided Backpropagation")
plt.show()


### Saliency Map (TensorFlow / Keras)

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

model = tf.keras.applications.ResNet50(weights="imagenet")

img = tf.keras.preprocessing.image.load_img(
    "image.jpg", target_size=(224, 224)
)
img = tf.keras.preprocessing.image.img_to_array(img)
img = tf.keras.applications.resnet.preprocess_input(img)
img = tf.convert_to_tensor(img[None, ...])

with tf.GradientTape() as tape:
    tape.watch(img)
    preds = model(img)
    class_idx = tf.argmax(preds[0])
    loss = preds[:, class_idx]

grads = tape.gradient(loss, img)
saliency = tf.reduce_max(tf.abs(grads), axis=-1)

plt.imshow(saliency[0], cmap="hot")
plt.axis("off")
plt.title("Saliency Map (TensorFlow)")
plt.show()


### Saliency Map Overlay on Original Image

In [ ]:
import numpy as np

sal = saliency[0].cpu().numpy()
sal = (sal - sal.min()) / (sal.max() - sal.min())

plt.imshow(img.resize((224, 224)))
plt.imshow(sal, cmap="jet", alpha=0.5)
plt.axis("off")
plt.title("Saliency Overlay")
plt.show()
